# Interpretable AI Framework for Lung Disease Classification from Chest X-Rays
**Team Member:** Md. Nadimul Islam Nadim (ID: 232-35-593)

Ei notebook ta apnar project outline অনুযায়ী তৈরি (Classification Backbone + Grad-CAM Explainability Layer + Evaluation).

**Kivabe use korben (Google Colab):**
1. Runtime -> Change runtime type -> **GPU (T4)** select korun.
2. Cell gulo upor theke niche, ek ek kore **Shift+Enter** diye run korun.
3. Kaggle dataset download korার jonne apnar Kaggle API key (`kaggle.json`) lagbe — সেই step এ বিস্তারিত instructions দেওয়া আছে।


## Step 0: Install & Import Libraries
Ei cell ta প্রথমে run করুন — দরকারি সব package install হবে।

In [ ]:
!pip install -q tensorflow scikit-learn matplotlib seaborn opencv-python-headless lime shap kaggle

import os, cv2, json, shutil, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## Step 1: Dataset Download (COVID-19 Radiography Database — Kaggle)

Apnar outline e je dataset gulo mention kora ache (NIH ChestX-ray14, CheXpert, COVID-19 Radiography Database, Shenzhen/Montgomery TB) — এখানে amra sobcheye shohoj o balanced **COVID-19 Radiography Database** use korchi, karon eita already 4-class labeled (Normal, COVID, Lung Opacity, Viral Pneumonia).

**Kaggle API key kivabe pabo:**
1. https://www.kaggle.com -> account -> Settings -> API -> "Create New Token" click korun.
2. `kaggle.json` file download hobe.
3. Niche cell run korle upload button ashbe — sei `kaggle.json` file ta upload korun.


In [ ]:
from google.colab import files
print("kaggle.json file ta upload korun:")
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)


In [ ]:

# Dataset download (~1-2 GB, koyek minute lagte pare)
!kaggle datasets download -d tawsifurrahman/covid19-radiography-database -p /content/data --unzip

In [ ]:
DATA_ROOT = "/content/data/COVID-19_Radiography_Dataset"
print(os.listdir(DATA_ROOT))


## Step 2: Data Preprocessing
- Image resize (224x224, DenseNet121 er jonne standard)
- Normalization (rescale 1/255)
- Augmentation: rotation, flip, zoom, contrast
- Train / Validation / Test split (70/15/15)


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
CLASSES = ["Normal", "COVID", "Lung_Opacity", "Viral Pneumonia"]

# Prottek class er "images" subfolder e chobi thake — ek jaygay collect kore CSV banano
rows = []
for c in CLASSES:
    folder = os.path.join(DATA_ROOT, c, "images")
    if not os.path.isdir(folder):
        folder = os.path.join(DATA_ROOT, c)
    for fname in os.listdir(folder):
        rows.append({"filepath": os.path.join(folder, fname), "label": c})

df = pd.DataFrame(rows)
print(df['label'].value_counts())

from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.9, 1.1]
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col="filepath", y_col="label", target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode="categorical")

val_gen = val_test_datagen.flow_from_dataframe(
    val_df, x_col="filepath", y_col="label", target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False)

test_gen = val_test_datagen.flow_from_dataframe(
    test_df, x_col="filepath", y_col="label", target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False)

# Class imbalance handle korার jonne class weights
from sklearn.utils.class_weight import compute_class_weight
labels_idx = train_gen.classes
class_weights = compute_class_weight('balanced', classes=np.unique(labels_idx), y=labels_idx)
class_weight_dict = dict(enumerate(class_weights))
print(class_weight_dict)


## Step 3: Classification Backbone (Transfer Learning — DenseNet121)
ImageNet pretrained DenseNet121, top e custom classification head add kora hoyeche.


In [ ]:
def build_model(num_classes, img_size=(224,224)):
    base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(*img_size, 3))
    base_model.trainable = False  # প্রথমে freeze — শুধু head train hobe

    inputs = Input(shape=(*img_size, 3))
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.4)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs)
    return model, base_model

model, base_model = build_model(num_classes=len(CLASSES))
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="categorical_crossentropy",
              metrics=["accuracy"])
model.summary()


## Step 4: Model Training
Duita phase e train kora hocche:
1. **Phase 1** — backbone freeze kore shudhu head train.
2. **Phase 2** — fine-tuning: backbone er kichu upper layer unfreeze kore chhoto learning rate diye pura model train.


In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("best_model_phase1.keras", monitor="val_accuracy", save_best_only=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)
]

history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weight_dict,
    callbacks=callbacks
)


In [ ]:
# Phase 2: Fine-tuning — backbone er last kichu layer unfreeze kora
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

callbacks_ft = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("best_model_finetuned.keras", monitor="val_accuracy", save_best_only=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)
]

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=callbacks_ft
)


In [ ]:
# Training curve plot
def plot_history(h1, h2):
    acc = h1.history['accuracy'] + h2.history['accuracy']
    val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss = h1.history['loss'] + h2.history['loss']
    val_loss = h1.history['val_loss'] + h2.history['val_loss']

    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(acc, label='Train Acc'); plt.plot(val_acc, label='Val Acc')
    plt.legend(); plt.title('Accuracy')
    plt.subplot(1,2,2)
    plt.plot(loss, label='Train Loss'); plt.plot(val_loss, label='Val Loss')
    plt.legend(); plt.title('Loss')
    plt.show()

plot_history(history1, history2)


## Step 5: Evaluation
Accuracy, Precision, Recall, F1-score, AUC-ROC, Confusion Matrix — outline er 6.5 section অনুযায়ী।


In [ ]:
test_loss, test_acc = model.evaluate(test_gen)
print(f"Test Accuracy: {test_acc*100:.2f}%")

y_pred_probs = model.predict(test_gen)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes
class_labels = list(test_gen.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_labels))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.show()


In [ ]:
# Multi-class AUC-ROC (One-vs-Rest)
from sklearn.preprocessing import label_binarize
y_true_bin = label_binarize(y_true, classes=list(range(len(class_labels))))
auc = roc_auc_score(y_true_bin, y_pred_probs, average='macro', multi_class='ovr')
print(f"Macro-average AUC-ROC: {auc:.4f}")


## Step 6: Explainability Layer — Grad-CAM
Grad-CAM heatmap dekhabe model chobir kon oncholer (lung region) upor base kore prediction diyeche.


In [ ]:
def get_gradcam_heatmap(model, img_array, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)

def overlay_gradcam(img_path, heatmap, alpha=0.4):
    img = cv2.imread(img_path)
    img = cv2.resize(img, IMG_SIZE)
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    overlayed = cv2.addWeighted(img, 1 - alpha, heatmap, alpha, 0)
    return cv2.cvtColor(overlayed, cv2.COLOR_BGR2RGB)

# DenseNet121 er shesh conv layer বের করা
last_conv_layer_name = None
for layer in reversed(base_model.layers):
    if len(layer.output_shape) == 4:
        last_conv_layer_name = layer.name
        break
print("Last conv layer:", last_conv_layer_name)


In [ ]:
def predict_and_explain(img_path):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    heatmap, pred_idx = get_gradcam_heatmap(model, img_array, last_conv_layer_name)
    overlayed = overlay_gradcam(img_path, heatmap)
    confidence = model.predict(img_array)[0][pred_idx]

    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1); plt.imshow(img); plt.title("Original X-Ray"); plt.axis('off')
    plt.subplot(1,2,2); plt.imshow(overlayed); plt.title(f"Grad-CAM: {class_labels[pred_idx]} ({confidence*100:.1f}%)"); plt.axis('off')
    plt.show()
    return class_labels[pred_idx], confidence

# Test set theke kichu sample image e try korun
sample_paths = test_df['filepath'].sample(3, random_state=1).tolist()
for p in sample_paths:
    predict_and_explain(p)


## Step 7 (Optional): LIME Explanation
Grad-CAM chhara, LIME diyeo local explanation dekhano jay — outline er 5.2 section e mention kora ache.


In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

def lime_explain(img_path):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0

    def predict_fn(images):
        return model.predict(images)

    explainer = lime_image.LimeImageExplainer()
    explanation = explainer.explain_instance(
        img_array, predict_fn, top_labels=1, hide_color=0, num_samples=500
    )
    temp, mask = explanation.get_image_and_mask(
        explanation.top_labels[0], positive_only=True, num_features=5, hide_rest=False
    )
    plt.imshow(mark_boundaries(temp, mask))
    plt.title("LIME Explanation")
    plt.axis('off')
    plt.show()

lime_explain(sample_paths[0])


## Step 8 (Optional): SHAP Explanation
SHAP relatively slow (deep model e), tai just 1-2 ta image e demo hisebe dekhano hocche.


In [ ]:
import shap

background = next(train_gen)[0][:10]  # background samples

explainer = shap.GradientExplainer(model, background)
sample_img = tf.keras.preprocessing.image.load_img(sample_paths[0], target_size=IMG_SIZE)
sample_arr = np.expand_dims(tf.keras.preprocessing.image.img_to_array(sample_img)/255.0, axis=0)

shap_values = explainer.shap_values(sample_arr)
shap.image_plot(shap_values, sample_arr)


## Step 9: Save Final Model

In [ ]:
model.save("interpretable_lung_disease_model.keras")
from google.colab import files
files.download("interpretable_lung_disease_model.keras")


## Next Step: User Interface (Streamlit)
Model train complete hoye gele, ekta shohoj Streamlit app diye deploy korte parben jekhane user chobi upload korle prediction + Grad-CAM heatmap dekhabe.
Eta local machine e alada `app.py` file hisebe run hoy (Colab e na) — separate `app.py` file diye dicchi, download kore local e `streamlit run app.py` diye chalaben.
